# TENOR-SAXS v2 — variable-resolution scattering demo

A hands-on, interactive demonstration of **TENOR-SAXS** (Technique for ENsemble Observation by Resolution variation in SAXS), from *"Variable-Resolution Scattering Reveals Ensemble Properties"* (Steinitz & Beck).

The method recovers a nanoparticle ensemble's polydispersity — the scattering-weighted relative variance of the radius of gyration, $V = \mathrm{Var}_w(R_g)/\langle R_g\rangle_w^2$ — directly from a **single 2D SAXS image**, by digitally smearing it with two different anisotropic point-spread functions and fitting the log-ratio of the two smeared images, without needing to know the instrument's own native resolution function.

Code: **https://github.com/roybeckbarkai/tenor-saxs-v2** (clean-room Python port, validated against the paper's MATLAB reference implementation — see `the internal validation notes` in the repo for every difference found and how it was resolved).

This notebook:
1. Installs the package directly from GitHub.
2. Provides an **interactive control panel** (dropdowns + input fields) to simulate a polydisperse ensemble and run the TENOR-SAXS protocol on it, with no code editing required.
3. Reports both the raw **apparent** (Guinier) $R_g$ and the **polydispersity-corrected** mean $R_g$ (derived analytically from the recovered $V$ — see the note in cell below).
4. Sweeps over a range of true $V$ values to show recovery accuracy.
5. Runs a small noise-sensitivity benchmark, reproducing the paper's own validation approach.

Runtime: a standard (CPU-only) Colab runtime is sufficient — nothing here needs a GPU.

In [ ]:
# --no-deps on the git install: Colab already ships numpy/scipy/pandas/matplotlib
# that satisfy our requirements, and force-reinstalling THOSE too (pulling a
# different numpy build than the one Colab's other preinstalled packages were
# built against) causes a numpy/BLAS ABI mismatch (AttributeError mentioning
# `_blas_supports_fpe` or similar on import). --force-reinstall --no-deps only
# touches the tiny tenor-saxs-v2 package itself, so Colab's own stack stays intact.
# If you still hit an import error after this cell, it means an OLDER numpy was
# already imported into this process from a previous run in this session --
# use Runtime > Restart session (not just re-running cells) and run this cell again.
%pip install -q --upgrade --force-reinstall --no-deps git+https://github.com/roybeckbarkai/tenor-saxs-v2.git
%pip install -q ipywidgets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from tenor_saxs_v2 import psf, simulation, protocol, benchmark, plotting, formfactors

np.set_printoptions(precision=4, suppress=True)
print("tenor_saxs_v2 imported OK")

## 1. Interactive control panel

Pick a form factor, distribution shape, mean radius of gyration $R_0$, true polydispersity $V$, and noise level, then click **Run** to simulate a 2D SAXS pattern and recover $V$/$R_g$ from it with TENOR-SAXS.

**Apparent vs. corrected $R_g$:** the direct Guinier-fit $R_g$ from a polydisperse ensemble is always biased *upward* — $R_{g,\mathrm{apparent}} = R_0\sqrt{1+V}$ (paper Eq. 5). Once TENOR-SAXS recovers $V$, that bias can be undone analytically: $R_{0,\mathrm{corrected}} = R_{g,\mathrm{apparent}}/\sqrt{1+V_{\mathrm{recovered}}}$ — the same first-order-in-$V$ relationship used throughout the method, requiring no assumption about which distribution shape actually produced the data (only $R_0$ and $V$ themselves enter, to this order). Both values are reported below.

In [ ]:
#@title ▶ TENOR-SAXS interactive control panel  (double-click, or the ⋮ menu, to view/edit the code) { display-mode: "form" }
FORM_FACTORS = {
    "Gaussian chain (\u03c6''=1/18)": "gaussian_chain",
    "Solid sphere (\u03c6''=-1/63)": "solid_sphere",
    "Spherical shell (\u03c6''=-1/45)": "spherical_shell",
    "Thin rod (\u03c6''=11/225)": "thin_rod",
}
DISTRIBUTIONS = ["normal", "lognormal", "schulz", "boltzmann", "triangular", "uniform"]
STRATEGIES = ["inverseVariance", "bestSingle", "mean", "median", "robust"]
WEIGHT_MODES = [("intensity \u2014 default: matches the paper and MATLAB, and is the statistically optimal choice under photon-counting noise", "intensity"),
                ("sqrt(intensity) \u2014 an earlier draft's alternative convention", "sqrt_intensity")]

style = {"description_width": "140px"}
layout = widgets.Layout(width="420px")

form_factor_dd = widgets.Dropdown(options=list(FORM_FACTORS.keys()), value="Gaussian chain (\u03c6''=1/18)", description="Form factor:", style=style, layout=layout)
distribution_dd = widgets.Dropdown(options=DISTRIBUTIONS, value="normal", description="Distribution:", style=style, layout=layout)
r0_input = widgets.FloatText(value=3.0, description="R0 (nm):", style=style, layout=layout)
v_slider = widgets.FloatSlider(value=0.08, min=0.0, max=0.5, step=0.01, description="True V:", style=style, layout=layout, readout_format=".3f")
n_radii_input = widgets.IntText(value=25, description="N radii:", style=style, layout=layout)
noise_mode_dd = widgets.Dropdown(options=["Noise-free", "Photon-counting noise"], value="Photon-counting noise", description="Noise mode:", style=style, layout=layout)
peak_photons_slider = widgets.FloatLogSlider(value=1e4, base=10, min=1, max=6, step=0.1, description="Peak photons:", style=style, layout=layout)
seed_input = widgets.IntText(value=0, description="Random seed:", style=style, layout=layout)

weight_mode_dd = widgets.Dropdown(options=WEIGHT_MODES, value="intensity", description="Fit weighting:", style=style, layout=layout)
strategy_dd = widgets.Dropdown(options=STRATEGIES, value="inverseVariance", description="Combine strategy:", style=style, layout=layout)

run_button = widgets.Button(description="Run simulation + analysis", button_style="success", icon="play", layout=widgets.Layout(width="260px", margin="12px 0 0 0"))
output_area = widgets.Output()

sim_box = widgets.VBox([widgets.HTML("<b>Simulation</b>"), form_factor_dd, distribution_dd, r0_input, v_slider, n_radii_input, noise_mode_dd, peak_photons_slider, seed_input])
analysis_box = widgets.VBox([widgets.HTML("<b>Analysis</b>"), weight_mode_dd, strategy_dd, run_button])
panel = widgets.HBox([sim_box, analysis_box])


def run_simulation_and_analysis(_button=None):
    with output_area:
        clear_output(wait=True)
        shape = FORM_FACTORS[form_factor_dd.value]
        phi2 = formfactors.GUINIER_TABLE[shape].phi2
        weight_power = formfactors.GUINIER_TABLE[shape].weight_power
        noise = 0 if noise_mode_dd.value == "Noise-free" else -abs(peak_photons_slider.value)
        rng = np.random.default_rng(seed_input.value)

        sim = simulation.scatter2d(
            rg=r0_input.value, noise=noise, v_rel=v_slider.value, phi2=phi2,
            det_pix=500, sd_dist=360.0, wavelength=0.1, det_side=3.5,
            psf0=psf.bartlett2d(3, 15), dist_type=distribution_dd.value,
            n_radii=n_radii_input.value, weight_power=weight_power, rng=rng,
        )
        r = protocol.tenor_protocol(
            sim.intensity, sim.qx, sim.qy, phi2=phi2,
            weight_mode=weight_mode_dd.value, strategy=strategy_dd.value,
        )

        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
        im = axes[0].imshow(np.log10(np.clip(sim.intensity, 1e-6, None)),
                             extent=[sim.qx.min(), sim.qx.max(), sim.qy.min(), sim.qy.max()], cmap="viridis")
        axes[0].set_xlabel("$q_x$ (nm$^{-1}$)"); axes[0].set_ylabel("$q_y$ (nm$^{-1}$)")
        axes[0].set_title("Simulated 2D SAXS pattern (log intensity)")
        fig.colorbar(im, ax=axes[0], label="$\\log_{10} I(q)$")

        names = list(r.v_estimates.keys())
        v_vals = [r.v_estimates[n] for n in names]
        v_ses = [r.v_se[n] for n in names]
        colors = ["tab:green" if r.status[n] == "ok" else "tab:red" for n in names]
        axes[1].errorbar(range(len(names)), v_vals, yerr=v_ses, fmt="o", capsize=4, ecolor="gray")
        for i, c in enumerate(colors):
            axes[1].plot(i, v_vals[i], "o", color=c)
        axes[1].axhline(v_slider.value, color="k", linestyle="--", label="true V")
        axes[1].axhline(r.best_v, color="tab:blue", linestyle=":", label=f"combined ({strategy_dd.value})")
        axes[1].set_xticks(range(len(names))); axes[1].set_xticklabels(names, rotation=30)
        axes[1].set_ylabel("V estimate"); axes[1].legend(fontsize=8)
        axes[1].set_title("Per-observable V estimates (green=ok, red=unusable)")
        fig.tight_layout()
        plt.show()

        print(f"{'':20s} {'true':>12s} {'recovered':>18s}")
        print(f"{'V':20s} {v_slider.value:12.4f} {r.best_v:12.4f} +/- {r.best_v_se:.4f}")
        print(f"{'Rg apparent':20s} {'--':>12s} {r.rg:12.4f}   (Guinier fit, biased upward by polydispersity)")
        print(f"{'Rg corrected':20s} {r0_input.value:12.4f} {r.rg_corrected:12.4f}   (bias-corrected using recovered V)")
        print()
        print("Per-observable detail:")
        for n in names:
            print(f"  {n:8s}: V={r.v_estimates[n]:.4f}  SE={r.v_se[n]:.4f}  status={r.status[n]!r}")


run_button.on_click(run_simulation_and_analysis)
display(panel, output_area)
run_simulation_and_analysis()  # run once with the defaults so the panel isn't empty

## 2. Recovery accuracy across a range of true $V$

Sweeping the true variance and comparing to what TENOR-SAXS recovers, noise-free — this reproduces (a small version of) the paper's own Fig. 2 validation.

In [ ]:
TRUE_RG = 3.0
PHI2 = formfactors.GUINIER_TABLE["gaussian_chain"].phi2
PSF0 = psf.bartlett2d(3, 15)

v_true_grid = np.linspace(0.0, 0.25, 11)
v_recovered = []
rg_apparent_list = []
rg_corrected_list = []
for v_true in v_true_grid:
    s = simulation.scatter2d(
        rg=TRUE_RG, noise=0, v_rel=v_true, phi2=PHI2,
        det_pix=500, sd_dist=360.0, wavelength=0.1, det_side=3.5,
        psf0=PSF0, dist_type="normal", n_radii=25,
    )
    r = protocol.tenor_protocol(s.intensity, s.qx, s.qy, phi2=PHI2)
    v_recovered.append(r.best_v)
    rg_apparent_list.append(r.rg)
    rg_corrected_list.append(r.rg_corrected)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(v_true_grid, v_true_grid, "k--", label="perfect recovery")
axes[0].plot(v_true_grid, v_recovered, "o-", label="TENOR-SAXS recovered")
axes[0].set_xlabel("true $V$"); axes[0].set_ylabel("recovered $V$")
axes[0].legend(); axes[0].set_title("Noise-free V recovery (Gaussian chain, R0=3nm)")

axes[1].axhline(TRUE_RG, color="k", linestyle="--", label="true R0")
axes[1].plot(v_true_grid, rg_apparent_list, "o-", label="apparent (Guinier)")
axes[1].plot(v_true_grid, rg_corrected_list, "s-", label="corrected")
axes[1].set_xlabel("true $V$"); axes[1].set_ylabel("$R_g$ (nm)")
axes[1].legend(); axes[1].set_title("Apparent vs. corrected Rg vs. true V")
fig.tight_layout()
plt.show()

## 3. Noise-sensitivity benchmark

A small-scale version of the paper's Fig. 6 noise-sensitivity study: sweep over a few $V$ values and photon-flux levels, with multiple noisy replicates each, and look at how the $\sqrt{V}$-recovery discrepancy shrinks as photon counts increase. This uses `tenor_saxs_v2.benchmark`, which also handles the (nontrivial) job of finding simulation inputs that realize a *scattering-weighted* target variance, since the discretizer only directly controls the *number*-weighted one.

This cell takes roughly 15-30 seconds on a standard Colab CPU runtime (reduce `n_cases`/`n_replicates` for a faster, coarser look).

In [ ]:
import time

config = benchmark.BenchmarkConfig(
    output_root="/content/tenor_benchmark_demo",
    n_replicates=10,
    v_values=np.array([0.01, 0.09, 0.25]),          # a few V values instead of the full 11
    peak_photons=np.array([1e3, 1e4, 1e5]),          # a few photon-flux levels instead of the full 6
)

t0 = time.time()
results_df, manifest = benchmark.run_benchmark(config)
print(f"Done in {time.time()-t0:.1f}s, {len(results_df)} rows, {results_df['Status'].eq('ok').mean():.0%} valid")

fig, axes = plotting.plot_tenor_benchmark_violins(results_df)
fig.suptitle("Noise-sensitivity demo (small-scale: 3 V values x 3 photon levels x 10 replicates)")
plt.show()

## Next steps

- Clone the repo (`!git clone https://github.com/roybeckbarkai/tenor-saxs-v2.git`) to run the full paper-figure reproduction scripts in `scripts/` (`reproduce_fig2.py`, `reproduce_fig3.py`, `reproduce_fig5_r0_sensitivity.py`, `reproduce_fig6_psf0_sensitivity.py`, `reproduce_fig6_noise_violin.py`, `reproduce_fig7_nradii_sensitivity.py`, `reproduce_fig8_pxn_sensitivity.py`), or the full 11-case x 6-noise-level x 30-replicate benchmark (`run_benchmark.py`).
- Load your own experimental 2D detector image by building `qx`/`qy` meshgrids from your beamline geometry and calling `protocol.tenor_protocol(intensity, qx, qy, phi2=...)` directly — it accepts any 2D intensity array with matching q-coordinate meshgrids.
- The `matlab/` folder contains the original MATLAB reference implementation and an Octave export harness (`run_reference_export.m`) used to validate this Python port numerically against it.